In [3]:
import cv2
import time,datetime
import imutils
import mediapipe as mp


In [28]:

cap=cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,1080)
cap.set(cv2.CAP_PROP_FPS,30)

hands=mp.solutions.hands.Hands(static_image_mode=True,max_num_hands=2,min_detection_confidence=0.5,min_tracking_confidence=0.5)
mpDraw=mp.solutions.drawing_utils
landmark_style=mpDraw.DrawingSpec(color=(255,255,255),thickness=2,circle_radius=4)
connection_style=mpDraw.DrawingSpec(color=(255,180,0),thickness=3,circle_radius=2)
prev_time=time.time()
num_frames=0
fps=0

while True:
    ret,frame=cap.read()

    if ret:
        # frame=cv2.rotate(frame,cv2.ROTATE_90_CCLOCKWISE)
        frame=cv2.flip(frame,-1)
        # frame=imutils.resize(frame,width=800)
        frame_rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

        num_frames+=1

        results=hands.process(frame_rgb)
        if results.multi_hand_landmarks:
            cv2.putText(frame,"Number of hands:"+str(len(results.multi_hand_landmarks)),(10,30),cv2.FONT_HERSHEY_PLAIN,3,(0,0,255),3)

            for i,handLms in enumerate(results.multi_hand_landmarks):
                mpDraw.draw_landmarks(frame,handLms,mp.solutions.hands.HAND_CONNECTIONS,landmark_style,connection_style)

                h,w,c=frame.shape
                x_list=[]
                y_list=[]

                for lm in handLms.landmark:
                    x_list.append(int(lm.x*w))
                    y_list.append(int(lm.y*h))

                x1,y1=max(min(x_list)-20,0),max(min(y_list)-20,0)
                x2,y2=min(max(x_list)+20,w),min(max(y_list)+20,h)

                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

                label="Hand"
                if results.multi_handedness:
                    label=results.multi_handedness[i].classification[0].label

                cv2.putText(frame,label,(x1,y1-20),cv2.FONT_HERSHEY_PLAIN,3,(255,0,255),3)

        # current_time=time.time()
        # elapsed_time=current_time-prev_time
        # if elapsed_time>=1:
        #     fps=num_frames/elapsed_time
        #     num_frames=0
        #     prev_time=current_time
        current_time=time.time()
        fps=1/(current_time-prev_time)
        prev_time=current_time

        cv2.putText(frame,f"FPS: {int(fps)}",(10,70),cv2.FONT_HERSHEY_PLAIN,3,(0,0,255),3)
        cv2.putText(frame,f"Number Frames: {num_frames}",(10,100),cv2.FONT_HERSHEY_PLAIN,3,(0,0,255),3)

        cv2.imshow("Hand Tracking",frame)
        # prev_time=current_time

        if cv2.waitKey(1)==ord("q"):
            break

    else:
        print("Camera error")
        break

cap.release()
hands.close()
cv2.destroyAllWindows()